# CodeAssistantAgent：本地工具与补丁流程

作者：minibeartwo。本 Notebook 是 Hello-Agents 共创项目的演示入口，复用桌面客户端的实际后端代码。

从上到下运行即可体验：环境检查 → 读取代码 → 提出补丁但不改文件 → 应用补丁 → 运行真实 unittest → 撤销。

这部分不调用模型，不需要 API Key。完整模型与桌面使用方式见 [README.md](README.md)。

In [ ]:
import sys
from pathlib import Path
from importlib.metadata import version

project = Path.cwd().resolve()
if not (project / "backend").is_dir():
    raise RuntimeError("请把 Notebook 的工作目录设为 minibeartwo-CodeAssistantAgent")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
assert sys.version_info >= (3, 11), "需要 Python 3.11+"
for package in ["hello-agents", "fastapi", "uvicorn"]:
    print(f"{package}: {version(package)}")
print("项目:", project)
print("Python:", sys.version.split()[0])

## 1. 在临时目录准备小项目

演示只操作系统临时目录，不修改示例或用户工作区。SQLite 存在临时工作区之外。最后一步会清理临时目录。

In [ ]:
from tempfile import TemporaryDirectory
from backend.workspace import Workspace, digest
from backend.store import Store, identifier
from backend.patches import Patches

sandbox = TemporaryDirectory(prefix="code-assistant-notebook-")
base = Path(sandbox.name)
root = base / "workspace"
root.mkdir()
# 直接构造源代码，保留 UTF-8 字节与行尾。
before = "def add(a, b):\n    return a - b\n"
after = "def add(a, b):\n    return a + b\n"
(root / "calculator.py").write_bytes(before.encode("utf-8"))
test_source = "import unittest\nfrom calculator import add\n\nclass AdditionTest(unittest.TestCase):\n    def test_add(self):\n        self.assertEqual(add(2, 3), 5)\n"
(root / "test_calculator.py").write_bytes(test_source.encode("utf-8"))
workspace = Workspace(str(root))
store = Store(base / "state.sqlite3")
workspace_id = identifier()
store.execute("INSERT INTO workspaces VALUES(?,?,?)", (workspace_id, str(root), "notebook-demo"))
patches = Patches(store)
print(workspace.tree())

## 2. 读取并生成待审阅补丁

文件的 hash 是版本基线。桌面 Agent 会把补丁保存为 proposed 后暂停；只有用户接受才会写入文件。

下面明确指定改动，演示补丁管理器，不把这段确定性代码当成模型生成结果。

In [ ]:
file = workspace.read("calculator.py")
print("文件:", file["path"])
print("基线:", file["hash"])
print(file["content"])
patch = patches.propose({"id": identifier(), "workspace_id": workspace_id}, [
    {"path": file["path"], "base_hash": file["hash"], "content": after}
])
assert workspace.read("calculator.py")["hash"] == digest(before)
print("补丁状态:", patch["status"], "；文件尚未改变")
print(patch["changes"][0]["diff"])

## 3. 应用与测试

这一格相当于在客户端点击“接受并应用”，并显式运行一次 unittest。命令执行器与客户端使用同一个模块，会展示实际退出码和输出。

仅在你决定应用上面改动时运行本格。

In [ ]:
from backend.commands import run_command

applied = patches.apply(patch["id"])
assert applied["status"] == "applied"
assert workspace.read("calculator.py")["hash"] == digest(after)

async def show_output(kind, data):
    if kind == "command.output":
        print(data["text"], end="")

result = await run_command(["python", "-m", "unittest", "discover", "-v"], root, show_output)
print("退出码:", result["exit_code"])
assert result["exit_code"] == 0
assert not result["timed_out"]

## 4. 撤销与清理

撤销同样校验版本。若文件在应用后又被外部编辑，系统会报告冲突并保留外部内容。这里没有外部改动，因此可以恢复。

In [ ]:
reverted = patches.apply(patch["id"], undo=True)
assert reverted["status"] == "reverted"
assert workspace.read("calculator.py")["hash"] == digest(before)
print("已恢复原始文件。")
sandbox.cleanup()
print("临时目录已清理。")

## 5. 使用真实模型和桌面客户端

在本项目目录执行 `start.ps1`，打开一个项目，在右上角配置模型地址、模型名和 API Key，然后提交代码任务。

桌面主进程负责启动 Python 服务和本机令牌；前端通过受控 IPC 调用 API。Agent 复用 HelloAgentsLLM 与 ToolRegistry，在有预算的循环中读取证据、提出补丁，等待审阅，再执行已授权测试。会话与事件保存在 SQLite。

[架构图](docs/diagrams/architecture.png) · [流程图](docs/diagrams/workflow.png) · [时序图](docs/diagrams/sequence.png)

后端与协议测试：`.venv/Scripts/python.exe -E -m pytest -q`。桌面集成测试：在 desktop 运行 `npm run build` 和 `npm run test:desktop`。